# Warehouse

Confirmation checks for the warehouse build. Jobs live in `spark/jobs/`.

## Spark session

In [1]:
# confirm connection with Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("warehouse")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    # without a cap this session holds every core
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    # one shared catalog on the mount, so tables created by spark-submit
    # resolve here too
    .config("spark.sql.warehouse.dir", "/opt/data/warehouse")
    .config(
        "javax.jdo.option.ConnectionURL",
        "jdbc:derby:;databaseName=/opt/data/metastore_db;create=true",
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

spark.range(5).selectExpr("id", "id * id AS squared").show()

Spark version : 3.5.0
Master        : spark://spark-master:7077
Application ID: app-20260803022541-0001
+---+-------+
| id|squared|
+---+-------+
|  0|      0|
|  1|      1|
|  2|      4|
|  3|      9|
|  4|     16|
+---+-------+



In [2]:
# confirm the data mount
from pathlib import Path

RAW = Path("/opt/data/raw")

print(f"raw root : {RAW}")
print(f"exists   : {RAW.is_dir()}")

for year in sorted(p for p in RAW.iterdir() if p.is_dir()):
    csvs = sorted(year.glob("*.csv"))
    print(f"{year.name}  {len(csvs):2d} csv")

raw root : /opt/data/raw
exists   : True
2019   4 csv
2020  12 csv
2021  12 csv
2022  11 csv
2023  12 csv
2024   1 csv
2025  12 csv


In [3]:
# confirm the executors can read off the mount, not just the driver
sample = spark.read.csv(
    "/opt/data/raw/2019/2019-Q1.csv", header=True, inferSchema=False
)

print(f"rows    : {sample.count():,}")
print(f"columns : {len(sample.columns)}")
sample.show(3, truncate=False)

rows    : 189,063
columns : 10
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|Trip Id|Trip  Duration|Start Station Id|Start Time      |Start Station Name      |End Station Id|End Time        |End Station Name                   |Bike Id|User Type    |
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|4581278|1547          |7021            |01/01/2019 00:08|Bay St / Albert St      |7233          |01/01/2019 00:33|King / Cowan Ave - SMART           |1296   |Annual Member|
|4581279|1112          |7160            |01/01/2019 00:10|King St W / Tecumseth St|7051          |01/01/2019 00:29|Wellesley St E / Yonge St (Green P)|2947   |Annual Member|
|4581280|589           |7055            |01/01/2019 00:15|Jarvis St / Carlton St  |7013          |0

## Stage table

Built by [`02_stage_table.py`](../jobs/02_stage_table.py). Empty until the
extract populates the `source_year` partitions.

In [4]:
# confirm stage table creation
WAREHOUSE = "/opt/data/warehouse"
STAGE_TRIPS_PATH = f"{WAREHOUSE}/stage_trips"

path = Path(STAGE_TRIPS_PATH)
partitions = sorted(p.name for p in path.glob("source_year=*"))
data_files = sorted(path.glob("**/*.parquet"))

print(f"path       : {path}")
print(f"exists     : {path.is_dir()}")
print(f"partitions : {', '.join(partitions) or '-'}")
print(f"data files : {len(data_files)}")

path       : /opt/data/warehouse/stage_trips
exists     : True
partitions : source_year=2019, source_year=2020, source_year=2021, source_year=2022, source_year=2023
data files : 16


In [5]:
# confirm the schema: 11 columns, all string
# read via the catalog - at zero rows there are no parquet files to infer from
stage = spark.table("stage_trips")

stage.printSchema()
print(f"columns : {len(stage.columns)}")
print(f"rows    : {stage.count():,}")

root
 |-- trip_id: string (nullable = true)
 |-- trip_duration: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- end_time: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- model: string (nullable = true)
 |-- source_year: string (nullable = true)

columns : 12
rows    : 18,920,187


In [6]:
# confirm the columns and their order
REFERENCE_COLUMNS = [
    "trip_id",
    "trip_duration",
    "start_time",
    "start_station_id",
    "start_station_name",
    "end_time",
    "end_station_id",
    "end_station_name",
    "bike_id",
    "user_type",
    "model",
]

actual = [f.name for f in stage.schema.fields if f.name != "source_year"]
non_string = [
    f.name
    for f in stage.schema.fields
    if f.name != "source_year" and f.dataType.simpleString() != "string"
]

print(f"missing    : {[c for c in REFERENCE_COLUMNS if c not in actual] or '-'}")
print(f"unexpected : {[c for c in actual if c not in REFERENCE_COLUMNS] or '-'}")
print(f"order      : {'ok' if actual == REFERENCE_COLUMNS else 'differs'}")
print(f"non-string : {non_string or '-'}")

missing    : -
unexpected : -
order      : ok
non-string : -


## Extract

Built by [`03_extract.py`](../jobs/03_extract.py). Raw csv -> stage table,
one partition per source year.

In [7]:
# confirm partitions and row counts per year
stage = spark.table("stage_trips")

stage.groupBy("source_year").count().orderBy("source_year").show()
print(f"total rows : {stage.count():,}")

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439007|
|       2020|2908652|
|       2021|3569543|
|       2022|4295984|
|       2023|5707001|
+-----------+-------+

total rows : 18,920,187


In [8]:
# confirm stage rows match the raw csv line count
from pathlib import Path

for year in [2019, 2020, 2021, 2022, 2023]:
    raw = 0
    for csv in sorted(Path(f"/opt/data/raw/{year}").glob("*.csv")):
        with open(csv, encoding="utf-8", errors="replace") as fh:
            raw += sum(1 for _ in fh) - 1  # minus header
    staged = stage.filter(stage.source_year == str(year)).count()
    flag = "ok" if raw == staged else f"MISMATCH ({raw - staged:+,})"
    print(f"{year}  raw {raw:>9,}  staged {staged:>9,}  {flag}")

2019  raw 2,439,517  staged 2,439,007  MISMATCH (+510)
2020  raw 2,911,308  staged 2,908,652  MISMATCH (+2,656)
2021  raw 3,575,182  staged 3,569,543  MISMATCH (+5,639)
2022  raw 4,300,240  staged 4,295,984  MISMATCH (+4,256)
2023  raw 5,713,141  staged 5,707,001  MISMATCH (+6,140)


In [9]:
# confirm no column arrived empty, which would mean a header mismatch
from pyspark.sql import functions as F

nulls = stage.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in stage.columns]
).collect()[0].asDict()

total = stage.count()
for col, n in nulls.items():
    print(f"{col:20s} {n:>9,} null  {n / total:6.1%}")

trip_id                      0 null    0.0%
trip_duration                0 null    0.0%
start_time                   0 null    0.0%
start_station_id             0 null    0.0%
start_station_name           0 null    0.0%
end_time                     0 null    0.0%
end_station_id               0 null    0.0%
end_station_name             0 null    0.0%
bike_id                      0 null    0.0%
user_type                    0 null    0.0%
model                        0 null    0.0%
source_year                  0 null    0.0%


In [10]:
# eyeball a few staged rows
stage.show(5, truncate=False)

+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|trip_id |trip_duration|start_time      |start_station_id|start_station_name                 |end_time        |end_station_id|end_station_name            |bike_id|user_type|model  |source_year|
+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|23529880|145          |08/01/2023 00:00|7101            |Lower Sherbourne St / The Esplanade|08/01/2023 00:02|7291          |190 Queens Quay E           |976    |casual   |UNKNOWN|2023       |
|26511369|593          |12/11/2023 09:00|7044            |Church St / Alexander St           |12/11/2023 09:09|7386          |D'Arcy St. /McCaul St. SMART|2483   |casual   |UNKNOWN|2023       |
|23931602|909          |08/15/

## Transform

Built by [`04_transform.py`](../jobs/04_transform.py). Invalid rows dropped,
non-critical columns repaired.

In [11]:
# confirm row counts per year after the clean
stage = spark.table("stage_trips")

stage.groupBy("source_year").count().orderBy("source_year").show()
print(f"total rows : {stage.count():,}")

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439007|
|       2020|2908652|
|       2021|3569543|
|       2022|4295984|
|       2023|5707001|
+-----------+-------+

total rows : 18,920,187


In [12]:
# confirm no invalid key values survived
from pyspark.sql import functions as F

TS = r"^[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}$"
INT = r"^[0-9]+$"

checks = {
    "trip_id not int": ~F.col("trip_id").rlike(INT),
    "duration <= 0": F.col("trip_duration").cast("double") <= 0,
    "start_time malformed": ~F.col("start_time").rlike(TS),
    "end_time malformed": ~F.col("end_time").rlike(TS),
    "start_station_id not int": ~F.col("start_station_id").rlike(INT),
    "end_station_id not int": ~F.col("end_station_id").rlike(INT),
    "literal 'NULL' name": (F.col("start_station_name") == "NULL")
    | (F.col("end_station_name") == "NULL"),
}

for label, cond in checks.items():
    n = stage.filter(cond).count()
    print(f"{label:26s} {n:>8,}  {'ok' if n == 0 else 'FAIL'}")

trip_id not int                   0  ok
duration <= 0                     0  ok
start_time malformed              0  ok
end_time malformed                0  ok
start_station_id not int          0  ok
end_station_id not int            0  ok
literal 'NULL' name               0  ok


In [13]:
# confirm no nulls remain in any column
nulls = stage.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in stage.columns]
).collect()[0].asDict()

for col, n in nulls.items():
    print(f"{col:20s} {n:>8,} null  {'ok' if n == 0 else 'FAIL'}")

trip_id                     0 null  ok
trip_duration               0 null  ok
start_time                  0 null  ok
start_station_id            0 null  ok
start_station_name          0 null  ok
end_time                    0 null  ok
end_station_id              0 null  ok
end_station_name            0 null  ok
bike_id                     0 null  ok
user_type                   0 null  ok
model                       0 null  ok
source_year                 0 null  ok


In [14]:
# confirm the normalized value sets
stage.groupBy("user_type").count().orderBy(F.desc("count")).show()
stage.groupBy("model").count().orderBy(F.desc("count")).show()
print(f"bike_id = -1 : {stage.filter(F.col('bike_id') == '-1').count():,}")

+---------+--------+
|user_type|   count|
+---------+--------+
|   casual|11053638|
|   annual| 7866549|
+---------+--------+

+-------+--------+
|  model|   count|
+-------+--------+
|UNKNOWN|18920187|
+-------+--------+

bike_id = -1 : 275


In [15]:
# confirm start_time parses month-first across the full range
parsed = stage.withColumn(
    "ts", F.to_timestamp("start_time", "MM/dd/yyyy HH:mm")
)

print(f"unparseable : {parsed.filter(F.col('ts').isNull()).count():,}")
parsed.select(F.min("ts").alias("min"), F.max("ts").alias("max")).show()
parsed.groupBy(F.month("ts").alias("month")).count().orderBy("month").show(12)

unparseable : 0
+-------------------+-------------------+
|                min|                max|
+-------------------+-------------------+
|2019-01-01 00:08:00|2023-12-31 23:59:00|
+-------------------+-------------------+

+-----+-------+
|month|  count|
+-----+-------+
|    1| 485786|
|    2| 433519|
|    3| 725796|
|    4|1066586|
|    5|1927430|
|    6|2382517|
|    7|2718662|
|    8|2871394|
|    9|2580833|
|   10|1992087|
|   11| 976197|
|   12| 759380|
+-----+-------+



In [16]:
stage.show(5, truncate=False)

+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|trip_id |trip_duration|start_time      |start_station_id|start_station_name                 |end_time        |end_station_id|end_station_name            |bike_id|user_type|model  |source_year|
+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|23529880|145          |08/01/2023 00:00|7101            |Lower Sherbourne St / The Esplanade|08/01/2023 00:02|7291          |190 Queens Quay E           |976    |casual   |UNKNOWN|2023       |
|26511369|593          |12/11/2023 09:00|7044            |Church St / Alexander St           |12/11/2023 09:09|7386          |D'Arcy St. /McCaul St. SMART|2483   |casual   |UNKNOWN|2023       |
|23931602|909          |08/15/

## Warehouse tables

Built by [`05_create_warehouse.py`](../jobs/05_create_warehouse.py). Four
dimensions and the fact table, empty until the load.

In [17]:
# confirm every table is registered
spark.sql("SHOW TABLES").show(truncate=False)

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|default  |dim_bike     |false      |
|default  |dim_station  |false      |
|default  |dim_time     |false      |
|default  |dim_user_type|false      |
|default  |fact_trip    |false      |
|default  |stage_trips  |false      |
+---------+-------------+-----------+



In [18]:
# confirm schemas and that each table starts empty
WAREHOUSE_TABLES = [
    "dim_time",
    "dim_station",
    "dim_bike",
    "dim_user_type",
    "fact_trip",
]

for name in WAREHOUSE_TABLES:
    df = spark.table(name)
    print(f"{name}  ({len(df.columns)} columns, {df.count():,} rows)")
    for f in df.schema.fields:
        print(f"    {f.name:26s} {f.dataType.simpleString()}")
    print()

dim_time  (9 columns, 0 rows)
    dim_time_id                timestamp
    dim_time_year              int
    dim_time_quarter           int
    dim_time_month             int
    dim_time_day               int
    dim_time_week              int
    dim_time_weekday           int
    dim_time_hour              int
    dim_time_minute            int

dim_station  (2 columns, 0 rows)
    dim_station_id             int
    dim_station_name           string

dim_bike  (2 columns, 0 rows)
    dim_bike_id                int
    dim_bike_model             string

dim_user_type  (2 columns, 0 rows)
    dim_user_type_id           int
    dim_user_type_name         string

fact_trip  (11 columns, 0 rows)
    fact_trip_id               bigint
    fact_trip_source_id        int
    fact_trip_duration         int
    fact_trip_start_time_id    timestamp
    fact_trip_end_time_id      timestamp
    fact_trip_start_station_id int
    fact_trip_end_station_id   int
    fact_trip_bike_id          int
 

In [19]:
# confirm the fact table partitioning
spark.sql("DESCRIBE TABLE fact_trip").show(30, truncate=False)

+--------------------------+---------+-------+
|col_name                  |data_type|comment|
+--------------------------+---------+-------+
|fact_trip_id              |bigint   |NULL   |
|fact_trip_source_id       |int      |NULL   |
|fact_trip_duration        |int      |NULL   |
|fact_trip_start_time_id   |timestamp|NULL   |
|fact_trip_end_time_id     |timestamp|NULL   |
|fact_trip_start_station_id|int      |NULL   |
|fact_trip_end_station_id  |int      |NULL   |
|fact_trip_bike_id         |int      |NULL   |
|fact_trip_user_type_id    |int      |NULL   |
|start_year                |int      |NULL   |
|start_month               |int      |NULL   |
|# Partition Information   |         |       |
|# col_name                |data_type|comment|
|start_year                |int      |NULL   |
|start_month               |int      |NULL   |
+--------------------------+---------+-------+

